<a href="https://colab.research.google.com/github/GeorgeGlennon/Part-II-Chemistry-Programming/blob/Exercise-4/Exercise4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
!git clone --branch Exercise-4 --single-branch https://github.com/GeorgeGlennon/Part-II-Chemistry-Programming.git /content/Part-II-Chemistry-Programming

Cloning into '/content/Part-II-Chemistry-Programming'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 59 (delta 12), reused 8 (delta 8), pack-reused 37 (from 1)
Receiving objects: 100% (59/59), 65.36 KiB | 2.61 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [32]:
from __future__ import annotations
from tabulate import tabulate
import numpy as np

def read_coordinates(filepath):
    """A function that reads a .xyz or .txt file and returns the molecular coordinates
    """

    with open(filepath, 'r') as file:
        lines = file.readlines()
        num_particles = int(lines[0].strip())  # Read number of particles
        coordinates = []

        num_particles = int(lines[0].strip())  # Read number of particles

        for line in lines[2:]:  # Skip the first two lines (header)
            parts = line.split()
            coords = list(map(float, parts[1:]))  # Convert x, y, z to float
            coordinates.append(coords)

    return num_particles, np.array(coordinates)

def Lennard_Jones_potential(r_vec: np.ndarray, epsilon=1.0, sigma=1.0) -> tuple[float, np.ndarray]:
    """
    Compute Lennard-Jones potential and force for a given vector position.

    Parameters:
    r_vec (np.array): Position vector between two particles.
    epsilon (float): Depth of potential well (default 1.0).
    sigma (float): Distance at which potential is zero (default 1.0).

    Returns:
    V (float): Lennard-Jones potential.
    F_vec (np.array): Force vector (negative gradient of potential).
    """
    r = np.linalg.norm(r_vec)  # Compute distance
    if r == 0:
        raise ValueError("Particles are at the same position, force is undefined.")

    # Compute Lennard-Jones potential
    sigma_r_6 = (sigma/r)**6
    sigma_r_12 = sigma_r_6**2
    V = 4 * epsilon * (sigma_r_12 - sigma_r_6)

    # Compute Lennard-Jones force magnitude
    F_mag = 24 * epsilon * ((2 * sigma_r_12) - sigma_r_6) / r

    # Compute force vector (F = F_mag * (r_vec / r))
    F_vec = -F_mag * (r_vec / r)

    return V, F_vec

def Morse_potential(r_vec: np.ndarray, D: float=1.0, sigma: float=1.0, r_eq: float=3.0) -> tuple[float, np.ndarray]:
    """
    Compute Morse potential and force for a given vector position.

    Parameters:
    r_vec (np.array): Position vector between two particles.
    D (float): Depth of the potential well (default 1.0).
    sigma (float): Controls the width of the potential well (default 1.0).
    r_eq (float): Equilibrium bond distance.

    Returns:
    V (float): Morse potential energy.
    F_vec (np.array): Force vector (negative gradient of potential).
    """
    r = np.linalg.norm(r_vec)  # Compute distance
    if r == 0:
        raise ValueError("Particles are at the same position, force is undefined.")

    # Compute Morse potential energy
    exp_term = np.exp(-(1/sigma) * (r - r_eq))
    V = D * (1 - exp_term)**2 - D

    # Compute Morse force magnitude
    F_mag = 2 * D * (1/sigma) * (1 - exp_term) * exp_term

    # Compute force vector (F = F_mag * (r_vec / r))
    F_vec = F_mag * (r_vec / r)

    return V, F_vec


class general_potential:
    """A class that performs clauclations on any given potential"""

    def __init__(self, potential):
        self.N, self.coords = read_coordinates("/content/Part-II-Chemistry-Programming/Random 7 atom coordinates.txt")
        self.dr = 1e-3
        self.potential = potential
        self.forces = np.zeros((self.N, 3))
        self.vectors = np.zeros((self.N, self.N, 3))


    def get_magnitudes(self):
        """A function that returns the magnitude of a given vector"""

        return np.linalg.norm(self.vectors, axis=-1)


    def total_energy(self) -> float:
        """A function that calculates the total energy of the system"""

        total = 0
        for i in range(self.N-1):
            for j in range(i+1, self.N):
                #print(self.get_magnitudes()[i][j], self.potential(self.get_magnitudes()[i][j]))
                total += (self.potential(self.get_magnitudes()[i][j]))[0]
        return total


    def update_vectors(self):
        self.vectors = self.coords[:, np.newaxis, :] - self.coords[np.newaxis, :, :]


    def update_forces(self):
        self.update_vectors()

        for i in range(self.N):
            self.forces[i] = 0
            for j in range(self.N):
                if i != j:
                    _, F_ij = self.potential(self.vectors[i, j])
                    self.forces[i] += F_ij  # Accumulate force on particle i


    def update_coordinates(self):
        # forces = np.array(self.get_forces())
        self.update_forces()
        self.coords -= self.forces * self.dr


    def iterate_to_equilibrium(self):
        n = 0
        while True:
            self.update_coordinates()
            if n % 1000 == 0:
                print(self.total_energy())
            if n == 100000:
                print(self.coords)
            n += 1
            if max(np.linalg.norm(f) for f in self.forces) <= (self.dr):
                break
        print(f"\nEquilibrium reached after {n} iterations with an energy of: {self.total_energy()} in energy units of this potential.\n")
        headers = ["Atom", "X", "Y", "Z"]
        data = [[f"H{i+1}", *self.coords[i]] for i in range(len(self.coords))]
        print(tabulate(data, headers=["Atom", "X", "Y", "Z"], tablefmt="grid", floatfmt=".6f"))
        return self.coords


    def export_coordinates(self, filepath, element="H"):
        """
        Exports molecular coordinates to a .xyz file.

        Parameters:
        - filepath (str): The output file path.
        - coordinates (np.array): Nx3 array of atomic coordinates.
        - element (str): The atomic element to use for all atoms (default: "H").
        """
        coordinates = self.iterate_to_equilibrium()
        num_atoms = len(coordinates)

        with open(filepath, "w") as file:
            file.write(f"{num_atoms}\n")  # Write the number of atoms
            file.write("Generated by export_coordinates\n")  # Comment line

            for coord in coordinates:
                file.write(f"{element} {coord[0]:.6f} {coord[1]:.6f} {coord[2]:.6f}\n")


if __name__ == "__main__":
    import time
    start = time.time()
    print("###A displacement-step of 1e-3 has been used for these potentials, a step of 1e-4 will give better convergence but will\ntake significantly longer than the 25 seconds a 1e-3 step takes to converge. This can be modified within the program.###\n")
    print(f"####Output from Lennard Jones potential####\n")
    general_potential(Lennard_Jones_potential).export_coordinates("/content/Lennard_Jones_output.xyz")
    print(f"\n####Output from Morse potential for re/sigma = 1.0####\n")
    general_potential(Morse_potential).export_coordinates("/content/Morse_output(re_by_sigma=1).xyz")
    print(f"\n####Output from Morse potential for re/sigma = 3.0####\n")
    general_potential(Morse_potential).export_coordinates("/content/Morse_output(re_by_sigma=3).xyz")
    print("\nThis program took", round(time.time() - start, 4), "seconds to converge.")

###A displacement-step of 1e-3 has been used for these potentials, a step of 1e-4 will give better convergence but will
take significantly longer than the 25 seconds a 1e-3 step takes to converge. This can be modified within the program.###

####Output from Lennard Jones potential####

-0.2608465430407378
-2.138901345524304
-3.248339549943331
-4.042037323981585
-4.04718113054878
-4.054840717749898
-4.067974634427284
-4.09809232622304
-4.315151313111514
-7.025988326001136
-7.0271835789887085
-7.02852922968442
-7.030058834431168
-7.031817331347746
-7.03386649350414
-7.036293934489453
-7.0392287550217825
-7.042870485763898
-7.047547056667188
-7.0538437299725905
-7.062934822558337
-7.077647115382847
-7.10748137156287
-7.237366627099701
-8.512878516666108
-12.318333546451365
-12.322501363510638
-12.32417986957766
-12.32623738496451
-12.32882829837567
-12.332207692191972
-12.336832100582617
-12.343615058206073
-12.35471918757658
-12.376978639684133
-12.452828133278969
-16.50524695240846

Equ